## Modelos

In [1]:
# Mistral

MISTRAL_SEM_CONTEXTO = "mistral:latest"
MISTRAL_COM_CONTEXTO = "mistral:latest"

# Mistral Instruct

#MISTRAL_I_SEM_CONTEXTO = "mistral:instruct"
#MISTRAL_I_COM_CONTEXTO = "mistral:instruct"

# LlAMA3

#LLAMA_SEM_CONTEXTO = "llama3"
#LLAMA_COM_CONTEXTO = "llama3"
API_PORTA = 11434       # Porta padrão do LLAMA

In [2]:
import requests

response = requests.get("http://localhost:11434/api/tags")

if response.status_code == 200:
    print("Ollama conectado!")
    for modelo in response.json()["models"]:
        print("-", modelo["name"])
else:
    print("Erro ao conectar ao Ollama.")

Ollama conectado!
- mistral:latest


## Bibliotecas

In [3]:
import os
import json
import pickle
import hashlib
import numpy as np
import faiss
import requests
from pathlib import Path
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

## Convertendo PDF'S

In [4]:
def extract_documents(path_doc="../Documentos"):
    folder = Path(path_doc)
    processed_docs = []

    if not folder.exists():
        print("O caminho informado não existe")
        return processed_docs

    pdf_files = list(folder.glob("*.pdf"))

    if not pdf_files:
        print("Nenhum arquivo pdf foi encontrado no caminho informado")
        return processed_docs

    print(f"Encontrados {len(pdf_files)} arquivos pdf.")

    for file in pdf_files:
        print(f"Arquivo selecionado: {file.name}")
        print("Lendo...")
        text = ""
        with open(file, 'rb') as f:
            reader = PdfReader(f)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
        processed_docs.append({
            "filename": file.name,
            "content": text
        })
        print("...Lido")
    return processed_docs

In [6]:
processed_docs = extract_documents()

print("\nExtração finalizada!")
print(f"Total de documentos processados: {len(processed_docs)}")

Encontrados 2 arquivos pdf.
Arquivo selecionado: Antiguidade Clássica — Grécia_ Pólis, Democracia, Cultura e Escravidão (1).pdf
Lendo...
...Lido
Arquivo selecionado: Antiguidade Clássica — Roma_ República, Império, Expansão e Crise.pdf
Lendo...
...Lido

Extração finalizada!
Total de documentos processados: 2


## Dividindo em Chunks

In [13]:
# 2. Dividir documentos em chunks
def dividir_em_chunks(documentos, tamanho_chunk=400, sobreposicao=100):
    todos_chunks = []

    for documento in documentos:
        texto = documento["content"]
        palavras = texto.split()

        i = 0

        while i < len(palavras):
            chunk = " ".join(palavras[i:i + tamanho_chunk])

            if chunk.strip():
                todos_chunks.append({
                    "filename": documento["filename"],
                    "content": chunk
                })

            i += tamanho_chunk - sobreposicao

    return todos_chunks

In [8]:
# 1. Extrair os documentos PDF
processed_docs = extract_documents()

# 2. Dividir os documentos em chunks
chunks = dividir_em_chunks(
    processed_docs,
    tamanho_chunk=400,
    sobreposicao=100
)

print(f"\nTotal de documentos: {len(processed_docs)}")
print(f"Total de chunks gerados: {len(chunks)}")

Encontrados 2 arquivos pdf.
Arquivo selecionado: Antiguidade Clássica — Grécia_ Pólis, Democracia, Cultura e Escravidão (1).pdf
Lendo...
...Lido
Arquivo selecionado: Antiguidade Clássica — Roma_ República, Império, Expansão e Crise.pdf
Lendo...
...Lido

Total de documentos: 2
Total de chunks gerados: 36


In [9]:

#Visualizando os chunks gerados

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i + 1}")
    print(f"Arquivo: {chunk['filename']}")
    print(f"Conteúdo: {chunk['content'][:300]}")
    print("-" * 60)


Chunk 1
Arquivo: Antiguidade Clássica — Grécia_ Pólis, Democracia, Cultura e Escravidão (1).pdf
Conteúdo: COMPÊNDIO DE HISTÓRIA GERAL · VOLUME II Antiguidade Clássica: A Civilização Grega A invenção da pólis, os fundamentos e limites da democracia ateniense, o modo de produção escravista e o legado cultural da Hélade “Nosso regime político não inveja as leis de nossos vizinhos; antes somos nós um exempl
------------------------------------------------------------

Chunk 2
Arquivo: Antiguidade Clássica — Grécia_ Pólis, Democracia, Cultura e Escravidão (1).pdf
Conteúdo: espaço geográfico fragmentado e marítimo. Situada no extremo sul da Península Balcânica e irradiada pelas ilhas dos mares Egeu e Jônico, pela costa ocidental da Ásia Menor (atual Turquia) e pelo sul da Península Itálica e da Sicília (a chamada Magna Grécia), a Grécia apresentava um relevo fortemente
------------------------------------------------------------

Chunk 3
Arquivo: Antiguidade Clássica — Grécia_ Pólis, Democr

# Embedding e indexação

In [6]:
# 3. Embeddings e indexação
def gerar_embeddings(chunks):
    print("Gerando embeddings e indexando...")

    # Inicializar modelo de embeddings
    embedder = SentenceTransformer(MODELO_EMBEDDING)

    # Extrair somente os conteúdos dos chunks
    textos = [chunk["content"] for chunk in chunks]

    # Gerar embeddings
    chunk_embeddings = embedder.encode(
        textos,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Criar índice FAISS
    index = faiss.IndexFlatL2(chunk_embeddings.shape[1])

    # Adicionar embeddings ao índice
    index.add(np.asarray(chunk_embeddings, dtype=np.float32))

    print("Indexação FAISS concluída!")
    print(f"Total de vetores indexados: {index.ntotal}")
    print(f"Dimensão dos embeddings: {chunk_embeddings.shape[1]}")

    return embedder, chunk_embeddings, index

In [11]:
# 1. Extrair documentos
#processed_docs = extract_documents()

# 2. Gerar chunks
#chunks = dividir_em_chunks(processed_docs)

# 3. Gerar embeddings e indexar
embedder, chunk_embeddings, index = gerar_embeddings(chunks)

Gerando embeddings e indexando...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexação FAISS concluída!
Total de vetores indexados: 36
Dimensão dos embeddings: 384


### Estrutura do processamento:
1. Chunks (Entrada) - Trechos textuais com seus respectivos nomes de arquivos
5. SentenceTransformer - Modelo all-MiniLM-L6-v2
7. Embeddings - Vetores numéricos de 384 dimensões
9. FAISS IndexFlatL2 (Indexação) - Armazenamento dos vetores para busca por distância euclidiana

# Função para buscar chunks mais relevantes

In [7]:
def buscar_trechos(pergunta, k=2):
    if not chunks:
        print("Nenhum chunk disponível para consulta.")
        return []

    # Garantir que k não ultrapasse a quantidade de chunks
    k = min(k, len(chunks))

    # Transformar a pergunta em embedding
    pergunta_emb = embedder.encode(
        [pergunta],
        convert_to_numpy=True
    )

    # Garantir o formato float32 exigido pelo FAISS
    pergunta_emb = np.asarray(pergunta_emb, dtype=np.float32)

    # Buscar os chunks mais próximos
    D, I = index.search(pergunta_emb, k)

    resultados = []

    for distancia, indice in zip(D[0], I[0]):
        if indice == -1:
            continue

        resultados.append({
            "filename": chunks[indice]["filename"],
            "content": chunks[indice]["content"],
            "distancia": float(distancia)
        })

    return resultados

In [13]:
pergunta = """A partir do século III d.C., o Império Romano entrou em um processo gradual de desagregação que
culminou na queda de sua porção ocidental em 476 d.C. Relacione o fim das guerras de conquista com a 
crise do sistema escravista e o processo de ruralização da economia romana."""

resultados = buscar_trechos(pergunta, k=2)

for i, resultado in enumerate(resultados):
    print(f"\nResultado {i + 1}")
    print(f"Documento: {resultado['filename']}")
    print(f"Distância: {resultado['distancia']:.4f}")
    print(f"Conteúdo:\n{resultado['content']}")
    print("-" * 60)


Resultado 1
Documento: Antiguidade Clássica — Roma_ República, Império, Expansão e Crise.pdf
Distância: 0.7100
Conteúdo:
Pax Romana e inaugurou o período do Baixo Império ou Dominato (235 – 476 d.C.). A raiz econômica fundamental dessa crise residiu no fim das guerras de conquista: ao atingir os limites defensáveis de suas fronteiras naturais (Reno, Danúbio, Eufrates e Saara), Roma deixou de expandir-se territorialmente, o que estancou a principal fonte de abastecimento de novos escravos e de saques provinciais. A escassez crescente de mão de obra escrava elevou seu preço, provocou a queda da produção agrícola e mineradora, desencadeou uma grave crise fiscal e levou o Estado a desvalorizar continuamente as moedas de prata e ouro, gerando hiperinflação e a retração generalizada do comércio e da vida urbana. Compêndio de História Geral · Antiguidade Clássica (Roma) Page 8 of 10 À crise econômica somou-se uma violenta anarquia militar e política (entre 235 e 284 d.C.), período no qual de

In [14]:
resultados = buscar_trechos(
    "Qual é o principal objetivo deste documento?",
    k=3
)

print(f"Total de resultados: {len(resultados)}")

Total de resultados: 3


#  Função para montar o prompt (contexto RAG)

In [8]:
def montar_prompt(pergunta, trechos):

    contexto = "\n\n".join([
        f"""
Fonte: {trecho['filename']}

Trecho utilizado:
{trecho['content']}
"""
        for trecho in trechos
    ])

    prompt = f"""
Você é um assistente de estudos especializado em responder
perguntas com base em apostilas e livros disponibilizados
pelo estudante.

Seu objetivo é fornecer respostas claras, corretas e fundamentadas
exclusivamente nos documentos recuperados pelo sistema.

CONTEXTO DOS DOCUMENTOS:
{contexto}

PERGUNTA DO ESTUDANTE:
{pergunta}

INSTRUÇÕES PARA A RESPOSTA:

1. Responda em português, utilizando linguagem clara e didática.
2. Utilize exclusivamente as informações presentes nos trechos
   fornecidos como contexto.
3. Explique o conteúdo de maneira organizada, preservando
   os conceitos e termos técnicos dos documentos.
4. Não invente informações, exemplos ou referências que
   não estejam fundamentados nos documentos.
5. Ao final da resposta, apresente as fontes utilizadas,
   informando o nome do arquivo correspondente.
6. Sempre que possível, indique os trechos específicos
   que fundamentaram a resposta.
7. Caso os documentos não contenham informações suficientes
   para responder, informe essa limitação claramente.
8. Não atribua informações a documentos que não as contenham.

FORMATO DA RESPOSTA:

Resposta:
[Explicação clara e didática da pergunta.]

Fontes utilizadas:
- [Nome do documento]
  Trecho: [Parte relevante utilizada na resposta]

Se não houver informações suficientes, explique isso
em vez de produzir uma resposta especulativa.
"""

    return prompt

In [16]:
pergunta = """
Relacione o fim das guerras de conquista com a crise
do sistema escravista e a ruralização da economia romana.
"""

trechos = buscar_trechos(pergunta, k=3)

for i, trecho in enumerate(trechos):
    print(f"\nTrecho {i + 1}")
    print(f"Documento: {trecho['filename']}")
    print(f"Distância: {trecho['distancia']:.4f}")
    print(f"Conteúdo: {trecho['content'][:500]}")
    print("-" * 60)

prompt = montar_prompt(pergunta, trechos)


Trecho 1
Documento: Antiguidade Clássica — Roma_ República, Império, Expansão e Crise.pdf
Distância: 0.7063
Conteúdo: vinha e a oliveira, cultivadas integralmente por mão de obra escrava. Arruinados pela longa ausência durante as guerras e incapazes de competir com o trigo barato importado das províncias (como Sicília e Egito) nem com a produção em larga escala das propriedades escravistas, dezenas de milhares de pequenos proprietários rurais perderam suas terras e migraram em massa para a cidade de Roma — fenômeno de intenso êxodo rural. Em Roma, essa multidão de camponeses expropriados, veteranos de guerra se
------------------------------------------------------------

Trecho 2
Documento: Antiguidade Clássica — Roma_ República, Império, Expansão e Crise.pdf
Distância: 0.7334
Conteúdo: Pax Romana e inaugurou o período do Baixo Império ou Dominato (235 – 476 d.C.). A raiz econômica fundamental dessa crise residiu no fim das guerras de conquista: ao atingir os limites defensáveis de s

#  Função genérica para perguntar ao modelo via Ollama API

In [9]:
def perguntar_ollama(prompt, modelo, porta=11434, max_tokens=512):
    url = f"http://localhost:11434/api/generate"

    try:
        response = requests.post(
            url,
            json={
                "model": modelo,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "num_predict": max_tokens,
                    "temperature": 0.2
                }
            },
            timeout=300
        )

        response.raise_for_status()

        resultado = response.json()

        return resultado.get(
            "response",
            "[ERRO] O modelo não retornou uma resposta."
        )

    except requests.exceptions.ConnectionError:
        return "[ERRO] Não foi possível conectar ao Ollama. Verifique se ele está em execução."

    except requests.exceptions.Timeout:
        return "[ERRO] O modelo demorou demais para responder."

    except requests.exceptions.HTTPError as erro:
        return f"[ERRO HTTP] {erro} - {response.text}"

    except Exception as erro:
        return f"[ERRO] {erro}"

### Criando o prompt sem contexto
Para comparar corretamente as respostas, precisamos garantir que o modelo sem RAG receba apenas a pergunta.

In [25]:
def perguntar_sem_contexto(pergunta, modelo=None):

    if modelo is None:
        modelo = MODELO_SEM_CONTEXTO

    prompt = f"""
Você é um assistente de estudos.

Responda à pergunta abaixo utilizando seus conhecimentos
adquiridos durante o treinamento.

Instruções:
- Responda em português.
- Utilize linguagem clara, didática e objetiva.
- Organize a resposta de maneira coerente.
- Não invente referências bibliográficas.

Pergunta:
{pergunta}

Resposta:
"""

    resposta = perguntar_ollama(
        prompt=prompt,
        modelo=modelo,
        porta=API_PORTA
    )

    return resposta

In [26]:
def perguntar_com_contexto(pergunta, modelo=None, k=3):

    if modelo is None:
        modelo = MODELO_COM_CONTEXTO

    # Busca os trechos relevantes nos documentos
    trechos = buscar_trechos(pergunta, k=k)

    if not trechos:
        return "Não foram encontrados trechos relevantes nos documentos."

    # Monta o prompt com o contexto recuperado
    prompt = montar_prompt(
        pergunta=pergunta,
        trechos=trechos
    )

    # Envia o prompt para o Ollama
    resposta = perguntar_ollama(
        prompt=prompt,
        modelo=modelo,
        porta=API_PORTA
    )

    return resposta

In [27]:
# 9. Comparação das respostas
pergunta_usuario = """
A partir do século III d.C., o Império Romano entrou em um processo gradual de desagregação que
culminou na queda de sua porção ocidental em 476 d.C. Relacione o fim das guerras de conquista com a 
crise do sistema escravista e o processo de ruralização da economia romana.
"""

modelos_teste = {
    "Mistral": "mistral:latest",
    "Mistral Instruct": "mistral:instruct",
    "Llama 3": "llama3"
}

for nome, modelo in modelos_teste.items():

    print("\n" + "=" * 80)
    print(f"MODELO: {nome}")
    print("=" * 80)

    print("\n--- SEM CONTEXTO RAG ---")

    resposta_sem = perguntar_sem_contexto(
        pergunta_usuario,
        modelo
    )

    print(resposta_sem)

    print("\n--- COM CONTEXTO RAG ---")

    resposta_com = perguntar_com_contexto(
        pergunta_usuario,
        modelo,
        k=3
    )

    print(resposta_com)


MODELO: Mistral

--- SEM CONTEXTO RAG ---
 A queda do Império Romano no século V d.C. foi o resultado de uma combinação de fatores complexos, entre eles a crise do sistema escravista, as guerras de conquista e a ruralização da economia romana.

As guerras de conquista foram um fator importante na decadência do Império Romano. A expansão territorial iniciada pelo Império Romano no século III d.C. atingiu seu ponto máximo, com o controle de um vasto território que se estendia desde a Britânia até a Síria. Porém, as guerras constantes exigiam grandes recursos financeiros e humanos, o que acabou esgotando o Império. Ao mesmo tempo, as fronteiras do Império se tornaram mais difíceis de manter, especialmente na fronteira do Danúbio, onde as tribos germânicas se tornaram uma ameaça constante.

A crise do sistema escravista também foi um fator importante na decadência do Império Romano. A escravidão era uma instituição central no sistema econômico romano, com escravos sendo utilizados em toda

# Configuração de salvando e checkpoint


In [21]:
PASTA_CACHE = Path("./cache_rag")

MODELO_EMBEDDING = "all-MiniLM-L6-v2"

TAMANHO_CHUNK = 400
SOBREPOSICAO = 100

In [22]:
def calcular_hash_arquivo(caminho):

    sha256 = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(4096), b""):
            sha256.update(bloco)

    return sha256.hexdigest()


def obter_fingerprints(pasta_documentos):

    pasta = Path(pasta_documentos)

    arquivos = sorted(pasta.glob("*.pdf"))

    fingerprints = {}

    for arquivo in arquivos:
        fingerprints[arquivo.name] = calcular_hash_arquivo(arquivo)

    return fingerprints

In [23]:
def carregar_ou_criar_base(path_doc="../Documentos"):

    global processed_docs
    global chunks
    global embedder
    global chunk_embeddings
    global index

    PASTA_CACHE.mkdir(parents=True, exist_ok=True)

    # Caminhos dos arquivos salvos
    arquivo_documentos = PASTA_CACHE / "documentos.pkl"
    arquivo_chunks = PASTA_CACHE / "chunks.pkl"
    arquivo_embeddings = PASTA_CACHE / "embeddings.npy"
    arquivo_indice = PASTA_CACHE / "indice.faiss"
    arquivo_metadata = PASTA_CACHE / "metadata.json"

    # Configurações utilizadas no processamento
    metadata_atual = {
        "embedding_model": MODELO_EMBEDDING,
        "tamanho_chunk": TAMANHO_CHUNK,
        "sobreposicao": SOBREPOSICAO,
        "documentos": obter_fingerprints(path_doc)
    }

    arquivos_cache = [
        arquivo_documentos,
        arquivo_chunks,
        arquivo_embeddings,
        arquivo_indice,
        arquivo_metadata
    ]

    # Verifica se todos os arquivos do cache existem
    cache_existe = all(
        arquivo.exists() for arquivo in arquivos_cache
    )

    # Verifica se as configurações e os PDFs continuam iguais
    cache_valido = False

    if cache_existe:

        with open(arquivo_metadata, "r", encoding="utf-8") as f:
            metadata_salvo = json.load(f)

        cache_valido = metadata_salvo == metadata_atual

    # ==================================================
    # CASO 1: CARREGAR CHECKPOINT
    # ==================================================

    if cache_valido:

        print("Checkpoint encontrado!")
        print("Carregando dados salvos...\n")

        with open(arquivo_documentos, "rb") as f:
            processed_docs = pickle.load(f)

        with open(arquivo_chunks, "rb") as f:
            chunks = pickle.load(f)

        chunk_embeddings = np.load(arquivo_embeddings)

        index = faiss.read_index(str(arquivo_indice))

        embedder = SentenceTransformer(MODELO_EMBEDDING)

        print("Base RAG carregada com sucesso!")
        print(f"Documentos: {len(processed_docs)}")
        print(f"Chunks: {len(chunks)}")
        print(f"Vetores: {index.ntotal}")

    # ==================================================
    # CASO 2: CRIAR NOVO CHECKPOINT
    # ==================================================

    else:

        print("Checkpoint inexistente ou desatualizado.")
        print("Iniciando processamento dos documentos...\n")

        # Leitura dos PDFs
        processed_docs = extract_documents(path_doc)

        if not processed_docs:
            raise ValueError("Nenhum documento foi encontrado.")

        # Divisão em chunks
        chunks = dividir_em_chunks(
            processed_docs,
            tamanho_chunk=TAMANHO_CHUNK,
            sobreposicao=SOBREPOSICAO
        )

        if not chunks:
            raise ValueError("Nenhum chunk foi gerado.")

        # Geração dos embeddings e índice FAISS
        embedder, chunk_embeddings, index = gerar_embeddings(chunks)

        # Salvando documentos
        with open(arquivo_documentos, "wb") as f:
            pickle.dump(processed_docs, f)

        # Salvando chunks
        with open(arquivo_chunks, "wb") as f:
            pickle.dump(chunks, f)

        # Salvando embeddings
        np.save(arquivo_embeddings, chunk_embeddings)

        # Salvando índice FAISS
        faiss.write_index(index, str(arquivo_indice))

        # Salvando metadados
        with open(arquivo_metadata, "w", encoding="utf-8") as f:
            json.dump(
                metadata_atual,
                f,
                ensure_ascii=False,
                indent=4
            )

        print("\nCheckpoint criado com sucesso!")
        print(f"Documentos: {len(processed_docs)}")
        print(f"Chunks: {len(chunks)}")
        print(f"Vetores: {index.ntotal}")

    return processed_docs, chunks, embedder, chunk_embeddings, index

In [24]:
processed_docs, chunks, embedder, chunk_embeddings, index = carregar_ou_criar_base()

Checkpoint encontrado!
Carregando dados salvos...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base RAG carregada com sucesso!
Documentos: 7
Chunks: 116
Vetores: 116
